# AI01 — SSL Swin V2 MAE Tiny trên Colab

Cùng layout/chức năng với `v1/ai01_colab_v1.ipynb`, nhưng dùng V2 Tiny, window 8.
Input đủ 11 Z, mặc định loss Z05/Z06/Z07 (zero-based), mask 75%.
Bật GPU. Notebook chỉ gọi code repo, không nhúng model/trainer. Run all chỉ smoke 3 batch.
Đọc `v2/README.md` trước khi train dài; chia well/patient trước SSL.

In [ ]:
import os, sys
from pathlib import Path
os.environ['USE_TF'] = '0'
REPO_DIR = Path('/content/SwinMAE_FJN')
if not REPO_DIR.exists():
    !git clone https://github.com/hahuy2004/SwinMAE_FJN.git "{REPO_DIR}"
os.chdir(REPO_DIR)
for name in ('v2/ai01_train_v2.py', 'v2/ai01_model_v2.py', 'utils/ai01_data.py', 'utils/ai01_requirements.txt'):
    assert Path(name).is_file(), f'Repo thiếu {name}; kiểm tra branch/commit đã push'
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
%pip install -q -r utils/ai01_requirements.txt
import torch
assert torch.cuda.is_available(), 'Bật GPU trong Runtime settings'
print(torch.cuda.get_device_name(0))

## Dữ liệu
Sửa ZIP_PATH hoặc dùng Prepared_Dataset đã giải nén ở root repo.
ZIP phải chứa Prepared_Dataset/train.txt và validation.txt, không chứa code.

In [ ]:
MOUNT_DRIVE = True
ZIP_PATH = Path('/content/drive/MyDrive/AI01/Prepared_Dataset.zip')
DATA_ROOT = REPO_DIR / 'Prepared_Dataset'
if not DATA_ROOT.exists():
    if MOUNT_DRIVE:
        from google.colab import drive
        drive.mount('/content/drive')
    import stat, zipfile
    from pathlib import PurePosixPath
    with zipfile.ZipFile(ZIP_PATH) as archive:
        for info in archive.infolist():
            p = PurePosixPath(info.filename)
            if p.is_absolute() or '..' in p.parts or '\\' in info.filename or not p.parts or p.parts[0] != 'Prepared_Dataset' or stat.S_ISLNK(info.external_attr >> 16):
                raise ValueError(f'ZIP không an toàn/đúng cấu trúc: {info.filename}')
        archive.extractall(REPO_DIR)
from utils.ai01_data import AI01UnlabelledDataset, ensure_disjoint_wells
train_dataset = AI01UnlabelledDataset(manifest=DATA_ROOT / 'train.txt', augment=False)
val_dataset = AI01UnlabelledDataset(manifest=DATA_ROOT / 'validation.txt', augment=False)
ensure_disjoint_wells(train_dataset, val_dataset)
print('Train:', len(train_dataset), 'Validation:', len(val_dataset), 'Input:', train_dataset[0].shape)

## Train smoke
Train dài: đổi EPOCHS, MAX_STEPS=0, warmup và chọn OUTPUT_DIR mới.
Mask giữ dense grid nên không giảm bộ nhớ theo tỷ lệ mask.

In [ ]:
OUTPUT_DIR = Path('output_ai01_v2_colab_smoke')
EPOCHS, MAX_STEPS, WARMUP_EPOCHS = 1, 3, 0
BATCH_SIZE, ACCUM_ITER = 1, 1
if OUTPUT_DIR.exists() and any(OUTPUT_DIR.iterdir()):
    raise FileExistsError('Chọn OUTPUT_DIR mới, không ghi đè run cũ')
!python -u -m v2.ai01_train_v2 --manifest Prepared_Dataset/train.txt --val_manifest Prepared_Dataset/validation.txt --output_dir "{OUTPUT_DIR}" --epochs {EPOCHS} --max_steps {MAX_STEPS} --warmup_epochs {WARMUP_EPOCHS} --batch_size {BATCH_SIZE} --accum_iter {ACCUM_ITER} --image_size 256 --window_size 8 --target_z 5 6 7 --gradient_checkpointing --device cuda
assert (OUTPUT_DIR / 'encoder-last.pt').is_file(), 'Train chưa hoàn tất'

In [ ]:
import json
history = [json.loads(line) for line in (OUTPUT_DIR / 'log.jsonl').read_text().splitlines() if line.strip()]
print(history[-1])
print('Encoder:', (OUTPUT_DIR / 'encoder-best.pt').resolve())
VERIFY_TRANSFER = False
if VERIFY_TRANSFER:
    %pip install -q -r utils/ai01_requirements_downstream.txt
    !python -m v2.ai01_verify_v2 --encoder "{OUTPUT_DIR / 'encoder-best.pt'}" --mask2former --device cuda

## Resume / evaluation / AI01
- Resume: giữ nguyên lệnh và tham số run cũ, thêm `--resume <run>/checkpoint-last.pt`.
- Evaluate: `python -m v2.ai01_evaluate_v2 --checkpoint <run>/checkpoint-last.pt --val_manifest Prepared_Dataset/validation.txt --output_dir <eval-folder-moi> --device cuda`.
- Downstream: `from v2.ai01_transfer_v2 import build_ai01_v2_mask2former, load_ai01_v2_encoder`. Phải dựng backbone V2, không nạp vào V1 bằng strict=False.
- Encoder-only exports không có decoder để tái tạo ảnh. Task heads cần supervised fine-tuning.
- Backup/download checkpoint trước khi Colab reset, hoặc đặt OUTPUT_DIR trực tiếp trên Drive.
- Smoke dữ liệu nhỏ không chứng minh chất lượng SSL hay segmentation.